# Medical Insurance Cost — Exploratory Data Analysis

This notebook explores the *Medical Cost Personal Dataset*, which contains demographic and lifestyle features for 1 338 insurance beneficiaries alongside their annual medical charges.


| Column | Type | Description |
|--------|------|-------------|
| `age` | int | Age of the primary beneficiary |
| `sex` | category | Biological sex (`male` / `female`) |
| `bmi` | float | Body Mass Index |
| `children` | int | Number of dependants covered |
| `smoker` | category | Smoking status (`yes` / `no`) |
| `region` | category | US residential region |
| `charges` | float | Individual medical costs billed (target) |

---

**Prerequisite:** Before running this notebook, make sure the raw data file exists at `data/insurance.csv`. If it doesn't, run the ingestion script from the project root first:
```bash
    python scripts/ingest_data.py
```


**Outputs:** All figures are automatically saved to `reports/figures/`. Running the final cell exports this notebook as a self-contained HTML report to `reports/eda_report.html`.

## 1. Setup

In [ ]:
import subprocess, sys
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="PiYG")
plt.rcParams["figure.dpi"] = 110

DATA_PATH = Path("..") / "data" / "insurance.csv"

# Output directories
FIGURES_DIR = Path("..") / "reports" / "figures"
REPORTS_DIR = Path("..") / "reports"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Data

In [ ]:
df = pd.read_csv(DATA_PATH)

# Casting categorical columns
for col in ["sex", "smoker", "region"]:
    df[col] = df[col].astype("category")

print(f"Shape: {df.shape}")
df.head()

## 3. Data Quality

In [ ]:
print("--- Data Types ---")
print(df.dtypes)
print()
print("--- Missing Values ---")
print(df.isnull().sum())
print()
print("--- Duplicate Rows ---")
print(df.duplicated().sum())

In [ ]:
df.describe(include="all").T

## 4. Target Variable: `charges`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["charges"], bins=40, kde=True, ax=axes[0])
axes[0].set_title("Distribution of charges")
axes[0].set_xlabel("charges ($)")

sns.histplot(np.log1p(df["charges"]), bins=40, kde=True, ax=axes[1])
axes[1].set_title("Distribution of log(charges)")
axes[1].set_xlabel("log(charges + 1)")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "charges_distribution.png", bbox_inches="tight")
plt.show()

print(f"Skewness (raw):  {df['charges'].skew():.3f}")
print(f"Skewness (log):  {np.log1p(df['charges']).skew():.3f}")

## 5. Numeric Features

In [ ]:
numeric_cols = ["age", "bmi", "children"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    sns.histplot(df[col], bins=30, kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "numeric_histograms.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    sns.boxplot(y=df[col], ax=ax, color="pink")
    ax.set_title(f"Boxplot of {col}")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "numeric_boxplots.png", bbox_inches="tight")
plt.show()

## 6. Categorical Features

In [ ]:
cat_cols = ["sex", "smoker", "region", "children"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax, palette="PiYG_r")
    ax.set_title(f"Count — {col}")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "categorical_counts.png", bbox_inches="tight")
plt.show()

## 7. Charges by Categorical Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, ["sex", "smoker", "region"]):
    sns.boxplot(data=df, x=col, y="charges", ax=ax, color="pink")
    ax.set_title(f"Charges by {col}")
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "charges_by_categorical.png", bbox_inches="tight")
plt.show()

In [ ]:
print("Mean charges by smoker status:")
print(df.groupby("smoker", observed=True)["charges"].mean().round(2))
print()
print("Mean charges by region:")
print(
    df.groupby("region", observed=True)["charges"]
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

## 8. Charges vs Numeric Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, numeric_cols):
    sns.scatterplot(data=df, x=col, y="charges", hue="smoker", alpha=0.55, ax=ax)
    ax.set_title(f"charges vs {col}")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "charges_vs_numeric.png", bbox_inches="tight")
plt.show()

## 9. Interaction: Age × BMI × Smoker

In [ ]:
g = sns.FacetGrid(df, col="smoker", height=5, aspect=1.1)
g.map_dataframe(sns.scatterplot, x="age", y="charges", hue="bmi", palette="Greens_d")
g.add_legend(title="BMI")
g.set_titles(col_template="Smoker = {col_name}")
plt.suptitle("Charges vs Age, coloured by BMI (split by smoker)", y=1.02)
g.savefig(FIGURES_DIR / "age_bmi_smoker_interaction.png", bbox_inches="tight")
plt.show()

## 10. Correlation Analysis

In [ ]:
df_encoded = df.copy()
df_encoded["sex"] = df_encoded["sex"].map({"male": 1, "female": 0})
df_encoded["smoker"] = df_encoded["smoker"].map({"yes": 1, "no": 0})
df_encoded = pd.get_dummies(df_encoded, columns=["region"], drop_first=True, dtype=int)

corr = df_encoded.corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="RdPu", vmin=-1, vmax=1, linewidths=0.5
)
plt.title("Correlation Matrix (lower triangle)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "correlation_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
print("Correlation with charges (descending):")
print(corr["charges"].sort_values(ascending=False).to_string())

## 11. Pairplot

In [ ]:
g = sns.pairplot(
    df,
    hue="smoker",
    vars=["age", "bmi", "children", "charges"],
    plot_kws={"alpha": 0.5},
    diag_kind="kde",
    palette="RdPu",
)
plt.suptitle("Pairplot of numeric features (hue = smoker)", y=1.01)
g.savefig(FIGURES_DIR / "pairplot.png", bbox_inches="tight")
plt.show()

## 12. Summary & Key Findings

### Dataset
- **1 338 records**, 7 columns, no missing values.
- Features: 3 numeric (`age`, `bmi`, `children`), 3 categorical (`sex`, `smoker`, `region`), 1 continuous target (`charges`).

### Target: `charges`
- Range: $1 121.87 – $63 770.43 · Mean: $13 270.42 · Median: $9 382.03
- Strongly right-skewed (skewness = 1.516). A log-transform brings it to near-normal (skewness = −0.09) - good for linear modelling.

### Feature Importance (correlation with `charges`)
| Feature | Pearson r |
|---------|-----------|
| `smoker` | **0.787** |
| `age` | 0.299 | 
| `bmi` | 0.198 |
| `children` | 0.068 |
| `sex` | 0.057 | 
| `region` | ≤ 0.074 |

### Key Insights
- **Smoking is the strongest predictor.** Smokers pay on average $32 050 vs $8 434 for non-smokers - nearly **4× more**.
- Age is the second predictor. Charges rise linearly with age and form three parallel bands: non-smokers (low), smokers with normal BMI (medium), smokers with high BMI (high).
- The average BMI is 30.66 (at the obesity boundary). High BMI (≥ 30) alone has moderate impact. Combined with smoking, it pushes charges to the highest tier.
- Sex has negligible impact. Males average $13 957 vs females $12 570 - a difference dwarfed by smoking status.
- Region has minimal impact. The southeast is the highest-cost region ($14 735) and the southwest the lowest ($12 347), a spread of just ~$2 400.
- Most beneficiaries have 0–2 children. The correlation with charges is small (r = 0.068).

In [ ]:
# --- Quick stats referenced in the summary above ---
print("Shape:", df.shape)
print("Duplicates:", df.duplicated().sum())
print()
print(
    f"Charges - mean: {df['charges'].mean():.2f} | median: {df['charges'].median():.2f} "
    f"| min: {df['charges'].min():.2f} | max: {df['charges'].max():.2f}"
)
print(
    f"Charges skewness - raw: {df['charges'].skew():.3f} | "
    f"log: {np.log1p(df['charges']).skew():.3f}"
)
print()
print("Mean charges by smoker:")
print(df.groupby("smoker", observed=True)["charges"].mean().round(2).to_string())
print()
print("Mean charges by sex:")
print(df.groupby("sex", observed=True)["charges"].mean().round(2).to_string())
print()
print("Mean charges by region:")
print(
    df.groupby("region", observed=True)["charges"]
    .mean()
    .sort_values(ascending=False)
    .round(2)
    .to_string()
)
print()
print("Age stats:")
print(df["age"].describe().round(2).to_string())
print()
print("BMI stats:")
print(df["bmi"].describe().round(2).to_string())
print()
print("Children value counts:")
print(df["children"].value_counts().sort_index().to_string())
print()
print("Correlation with charges (descending):")
print(corr["charges"].sort_values(ascending=False).to_string())

In [ ]:
notebook_path = Path(__file__) if "__file__" in dir() else Path("eda.ipynb")
notebook_path = Path("eda.ipynb")
output_path = REPORTS_DIR / "eda_report.html"

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "nbconvert",
        "--to",
        "html",
        "--output",
        str(output_path.resolve()),
        str(notebook_path.resolve()),
    ],
    capture_output=True,
    text=True,
)

if result.returncode == 0:
    print(f"HTML report exported to {output_path.resolve()}")
else:
    print("nbconvert failed:")
    print(result.stderr)